# Social Media & Content Analytics — Exploratory Data Analysis
**Author:** Francisco Costa  
**Period:** January 2023 – December 2024  
**Purpose:** Pre-dashboard exploratory analysis covering platform performance, content insights, campaign ROI, customer behaviour, and churn prediction.

## 1. Setup & Data Loading

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (13, 5), 'font.family': 'Arial',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlepad': 12,
})

PURPLE='#4A0E8F'; TEAL='#00897B'; AMBER='#F57F17'
RED='#B71C1C'; GREEN='#1B5E20'; NAVY='#1A237E'

DB_PATH = 'social_media_analytics.db'  # update path if needed
conn = sqlite3.connect(DB_PATH)
posts       = pd.read_sql('SELECT * FROM Posts',                conn)
eng         = pd.read_sql('SELECT * FROM Engagement',           conn)
camps       = pd.read_sql('SELECT * FROM Campaigns',            conn)
perf        = pd.read_sql('SELECT * FROM Campaign_Performance', conn)
customers   = pd.read_sql('SELECT * FROM Customers',            conn)
orders      = pd.read_sql('SELECT * FROM Orders',               conn)
products    = pd.read_sql('SELECT * FROM Products',             conn)
audience    = pd.read_sql('SELECT * FROM Audience',             conn)
influencers = pd.read_sql('SELECT * FROM Influencers',          conn)
conn.close()

for df, cols in [(posts,['PostDate']),(orders,['OrderDate']),(customers,['JoinDate']),
                 (audience,['Date']),(perf,['Date'])]:
    for c in cols:
        df[c] = pd.to_datetime(df[c])

posts['Year']  = posts['PostDate'].dt.year
posts['Month'] = posts['PostDate'].dt.month
orders['Year'] = orders['OrderDate'].dt.year
orders['Month']= orders['OrderDate'].dt.month

print('Data loaded successfully')
for name, df in [('Posts',posts),('Engagement',eng),('Campaigns',camps),
                 ('Campaign_Performance',perf),('Customers',customers),
                 ('Orders',orders),('Products',products),
                 ('Audience',audience),('Influencers',influencers)]:
    print(f'  {name:<25} {len(df):>5} rows')


ModuleNotFoundError: No module named 'sklearn'

## 2. Data Quality Check

In [ ]:
for name, df in [('Posts',posts),('Engagement',eng),('Orders',orders),
                 ('Customers',customers),('Products',products)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f'{name}: ', end='')
    if len(nulls) == 0:
        print('Clean — no nulls')
    else:
        for col, n in nulls.items():
            print(f'{col}={n}', end=' ')
        print()


## 3. Platform Performance Analysis

In [ ]:
df = posts.merge(eng, on='PostID', suffixes=('_post','_eng'))

platform = df.groupby('Platform_post').agg(
    Posts=('PostID','count'),
    TotalReach=('Reach','sum'),
    AvgReach=('Reach','mean'),
    AvgEngRate=('EngagementRate','mean'),
    TotalClicks=('Clicks','sum'),
    TotalShares=('Shares','sum'),
    AvgVirScore=('VirScore','mean')
).sort_values('TotalReach', ascending=False).round(2)
print(platform.to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = [PURPLE, TEAL, AMBER, NAVY, GREEN, RED]
plats = platform.sort_values('TotalReach', ascending=True)

axes[0].barh(plats.index, plats['TotalReach']/1e6, color=colors[:len(plats)])
axes[0].set_title('Total Reach by Platform (M)')
for i, v in enumerate(plats['TotalReach'].values/1e6):
    axes[0].text(v+0.1, i, f'{v:.1f}M', va='center', fontsize=9)

plats_eng = platform.sort_values('AvgEngRate', ascending=True)
axes[1].barh(plats_eng.index, plats_eng['AvgEngRate'],
             color=[colors[list(platform.index).index(p)] for p in plats_eng.index])
axes[1].set_title('Avg Engagement Rate (%)')
for i, v in enumerate(plats_eng['AvgEngRate'].values):
    axes[1].text(v+0.05, i, f'{v:.2f}%', va='center', fontsize=9)

plats_posts = platform.sort_values('Posts', ascending=True)
axes[2].barh(plats_posts.index, plats_posts['Posts'], color=colors[:len(plats_posts)])
axes[2].set_title('Post Volume by Platform')
for i, v in enumerate(plats_posts['Posts'].values):
    axes[2].text(v+2, i, str(v), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_01_platform_performance.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Content Type Performance

In [ ]:
content = df.groupby('ContentType').agg(
    Posts=('PostID','count'),
    AvgReach=('Reach','mean'),
    AvgEngRate=('EngagementRate','mean'),
    AvgCTR=('CTR','mean'),
    TotalShares=('Shares','sum'),
    TotalSaves=('Saves','sum'),
    TotalVideoViews=('VideoViews','sum')
).sort_values('AvgEngRate', ascending=False).round(2)
print(content.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

content_platform = df.groupby(['Platform_post','ContentType'])['EngagementRate'].mean().unstack()
sns.heatmap(content_platform.round(2), annot=True, fmt='.2f',
            cmap='RdYlGn', ax=axes[0], linewidths=0.5,
            cbar_kws={'label': 'Avg Engagement Rate (%)'})
axes[0].set_title('Engagement Rate: Platform x Content Type')

content_sorted = content.sort_values('AvgEngRate', ascending=True)
axes[1].barh(content_sorted.index, content_sorted['AvgEngRate'],
             color=[PURPLE if v > content['AvgEngRate'].mean() else '#E0E0E0'
                    for v in content_sorted['AvgEngRate']])
axes[1].axvline(content['AvgEngRate'].mean(), color=RED, linestyle='--',
                linewidth=1.5, label=f"Avg: {content['AvgEngRate'].mean():.2f}%")
axes[1].set_title('Avg Engagement Rate by Content Type')
axes[1].legend()
for i, v in enumerate(content_sorted['AvgEngRate'].values):
    axes[1].text(v+0.05, i, f'{v:.2f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_02_content_type.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Campaign ROI Analysis

In [ ]:
camp_merged = camps.merge(perf, on='CampaignID')
by_platform = camp_merged.groupby('Platform_x').agg(
    Campaigns=('CampaignID','nunique'),
    TotalSpend=('Spend','sum'),
    TotalRevenue=('Revenue','sum'),
    TotalConversions=('Conversions','sum'),
    AvgCTR=('CTR','mean'),
    AvgConvRate=('ConvRate','mean')
).round(2)
by_platform['ROAS'] = (by_platform['TotalRevenue']/by_platform['TotalSpend']).round(2)
by_platform['CPA']  = (by_platform['TotalSpend']/by_platform['TotalConversions']).round(2)
by_platform = by_platform.sort_values('ROAS', ascending=False)
print(by_platform.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plat_sorted = by_platform.sort_values('ROAS', ascending=True)
bar_colors = [GREEN if r>=2 else AMBER if r>=1 else RED for r in plat_sorted['ROAS'].values]
axes[0].barh(plat_sorted.index, plat_sorted['ROAS'], color=bar_colors)
axes[0].axvline(x=1, color=RED, linestyle='--', linewidth=1.5, label='Break-even (1.0x)')
axes[0].axvline(x=2, color=GREEN, linestyle='--', linewidth=1.5, label='Target (2.0x)')
for i, v in enumerate(plat_sorted['ROAS'].values):
    axes[0].text(v+0.02, i, f'{v:.2f}x', va='center', fontsize=9)
axes[0].set_title('ROAS by Platform')
axes[0].legend(fontsize=8)

by_obj = camp_merged.groupby('Objective').agg(
    TotalSpend=('Spend','sum'),
    TotalRevenue=('Revenue','sum'),
    TotalConversions=('Conversions','sum')
)
by_obj['ROAS'] = (by_obj['TotalRevenue']/by_obj['TotalSpend']).round(2)
by_obj_s = by_obj.sort_values('ROAS', ascending=True)
bar_colors2 = [GREEN if r>=2 else AMBER if r>=1 else RED for r in by_obj_s['ROAS'].values]
axes[1].barh(by_obj_s.index, by_obj_s['ROAS'], color=bar_colors2)
axes[1].axvline(x=1, color=RED, linestyle='--', linewidth=1.5, label='Break-even')
axes[1].axvline(x=2, color=GREEN, linestyle='--', linewidth=1.5, label='Target (2.0x)')
for i, v in enumerate(by_obj_s['ROAS'].values):
    axes[1].text(v+0.02, i, f'{v:.2f}x', va='center', fontsize=9)
axes[1].set_title('ROAS by Campaign Objective')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('chart_03_campaign_roas.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
camp_agg = camp_merged.groupby('CampaignID').agg(
    TotalSpend=('Spend','sum'),
    TotalRevenue=('Revenue','sum'),
    Platform=('Platform_x','first')
).reset_index()
camp_agg['ROAS'] = camp_agg['TotalRevenue'] / camp_agg['TotalSpend'].replace(0, np.nan)

platform_colors = {'Instagram':PURPLE,'TikTok':TEAL,'LinkedIn':NAVY,
                   'YouTube':RED,'Facebook':AMBER,'X (Twitter)':GREEN}

fig, ax = plt.subplots(figsize=(12, 6))
for plat, color in platform_colors.items():
    mask = camp_agg['Platform'] == plat
    ax.scatter(camp_agg[mask]['TotalSpend'], camp_agg[mask]['TotalRevenue'],
               c=color, label=plat, alpha=0.7, s=60)
max_val = max(camp_agg['TotalSpend'].max(), camp_agg['TotalRevenue'].max())
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1, alpha=0.4, label='Break-even')
ax.set_title('Campaign Spend vs Revenue — All Campaigns')
ax.set_xlabel('Total Spend (EUR)')
ax.set_ylabel('Total Revenue (EUR)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('chart_04_spend_vs_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Campaigns above break-even: {(camp_agg['ROAS'] >= 1).sum()}")
print(f"Campaigns below break-even: {(camp_agg['ROAS'] < 1).sum()}")
print(f"Overall ROAS: {camp_agg['TotalRevenue'].sum()/camp_agg['TotalSpend'].sum():.2f}x")


## 6. Audience Growth Analysis

In [ ]:
growth = audience.groupby('Platform').agg(
    StartFollowers=('TotalFollowers','min'),
    EndFollowers=('TotalFollowers','max'),
    TotalNew=('NewFollowers','sum'),
    TotalLost=('LostFollowers','sum'),
    AvgGrowthRate=('GrowthRatePct','mean')
).round(2)
growth['NetGrowth'] = growth['EndFollowers'] - growth['StartFollowers']
growth['GrowthPct'] = ((growth['EndFollowers']-growth['StartFollowers'])/growth['StartFollowers']*100).round(1)
growth = growth.sort_values('EndFollowers', ascending=False)
print(growth.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for plat, color in platform_colors.items():
    d = audience[audience['Platform']==plat].sort_values('Date')
    axes[0].plot(d['Date'], d['TotalFollowers']/1000, marker='o', markersize=3,
                 label=plat, color=color, linewidth=2)
axes[0].set_title('Follower Growth by Platform (000s)')
axes[0].set_ylabel('Total Followers (000s)')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=30)

growth_sorted = growth.sort_values('GrowthPct', ascending=True)
bar_colors = [platform_colors.get(p, PURPLE) for p in growth_sorted.index]
axes[1].barh(growth_sorted.index, growth_sorted['GrowthPct'], color=bar_colors)
for i, v in enumerate(growth_sorted['GrowthPct'].values):
    axes[1].text(v+0.3, i, f'{v:.1f}%', va='center', fontsize=9)
axes[1].set_title('Total Follower Growth % (2023-2024)')

plt.tight_layout()
plt.savefig('chart_05_audience_growth.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Customer Behaviour & Revenue

In [ ]:
cust_orders = customers.merge(orders, on='CustomerID', how='left')

by_channel = cust_orders.groupby('AcqChannel').agg(
    Customers=('CustomerID','nunique'),
    Orders=('OrderID','count'),
    TotalRevenue=('Revenue','sum'),
    AvgOrderValue=('Revenue','mean'),
    TotalMargin=('Margin','sum')
).round(2)
by_channel['CLV'] = (by_channel['TotalRevenue']/by_channel['Customers']).round(2)
by_channel['OrdersPerCust'] = (by_channel['Orders']/by_channel['Customers']).round(2)
by_channel = by_channel.sort_values('TotalRevenue', ascending=False)
print(by_channel.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

chan_sorted = by_channel.sort_values('CLV', ascending=True)
axes[0].barh(chan_sorted.index, chan_sorted['CLV'], color=PURPLE)
axes[0].axvline(by_channel['CLV'].mean(), color=RED, linestyle='--',
                linewidth=1.5, label=f"Avg CLV: EUR{by_channel['CLV'].mean():.0f}")
for i, v in enumerate(chan_sorted['CLV'].values):
    axes[0].text(v+5, i, f'EUR{v:.0f}', va='center', fontsize=9)
axes[0].set_title('Customer Lifetime Value by Acquisition Channel')
axes[0].legend()

by_seg = cust_orders.groupby('Segment').agg(
    TotalRevenue=('Revenue','sum'),
    Customers=('CustomerID','nunique')
).sort_values('TotalRevenue', ascending=True)
seg_colors = [GREEN if s=='Champion' else TEAL if s=='Loyal'
              else AMBER if s in ('New','Potential') else RED
              for s in by_seg.index]
axes[1].barh(by_seg.index, by_seg['TotalRevenue']/1000, color=seg_colors)
for i, v in enumerate(by_seg['TotalRevenue'].values/1000):
    axes[1].text(v+0.5, i, f'EUR{v:.0f}K', va='center', fontsize=9)
axes[1].set_title('Total Revenue by Customer Segment (KEUR)')

plt.tight_layout()
plt.savefig('chart_06_customer_revenue.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
monthly = orders.groupby(['Year','Month']).agg(
    Orders=('OrderID','count'),
    Revenue=('Revenue','sum'),
    AOV=('Revenue','mean')
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
yr_colors = {2023: PURPLE, 2024: TEAL}

for yr in [2023, 2024]:
    d = monthly[monthly['Year']==yr].set_index('Month')
    axes[0].plot(d.index, d['Revenue']/1000, marker='o', label=str(yr),
                 color=yr_colors[yr], linewidth=2.5)
    axes[1].plot(d.index, d['AOV'], marker='o', label=str(yr),
                 color=yr_colors[yr], linewidth=2.5)

axes[0].set_title('Monthly Revenue Trend by Year (KEUR)')
axes[0].set_ylabel('Revenue (000s EUR)')
axes[0].legend()
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(month_labels)

axes[1].set_title('Monthly Average Order Value by Year (EUR)')
axes[1].set_ylabel('Avg Order Value (EUR)')
axes[1].legend()
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(month_labels)

plt.tight_layout()
plt.savefig('chart_07_revenue_trend.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Product Performance

In [ ]:
prod_orders = orders.merge(products, on='ProductID')
by_cat = prod_orders.groupby('Category').agg(
    Orders=('OrderID','count'),
    UnitsSold=('Quantity','sum'),
    Revenue=('Revenue','sum'),
    Margin=('Margin','sum'),
    AvgPrice=('UnitPrice','mean')
).round(2)
by_cat['MarginPct'] = (by_cat['Margin']/by_cat['Revenue']*100).round(1)
by_cat = by_cat.sort_values('Revenue', ascending=False)
print(by_cat.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

cat_sorted = by_cat.sort_values('Revenue', ascending=True)
axes[0].barh(cat_sorted.index, cat_sorted['Revenue']/1000, color=PURPLE, label='Revenue')
axes[0].barh(cat_sorted.index, cat_sorted['Margin']/1000, color=TEAL, label='Margin', alpha=0.8)
axes[0].set_title('Revenue vs Margin by Category (KEUR)')
axes[0].legend()

cat_margin = by_cat.sort_values('MarginPct', ascending=True)
bar_colors = [GREEN if m>=40 else AMBER if m>=30 else RED for m in cat_margin['MarginPct'].values]
axes[1].barh(cat_margin.index, cat_margin['MarginPct'], color=bar_colors)
axes[1].axvline(by_cat['MarginPct'].mean(), color=NAVY, linestyle='--',
                linewidth=1.5, label=f"Avg: {by_cat['MarginPct'].mean():.1f}%")
axes[1].set_title('Margin % by Product Category')
axes[1].legend()
for i, v in enumerate(cat_margin['MarginPct'].values):
    axes[1].text(v+0.2, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('chart_08_product_performance.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Influencer Performance

In [ ]:
tier_order = ['Nano','Micro','Mid-tier','Macro','Mega']
by_tier = influencers.groupby('Tier').agg(
    Count=('InfluencerID','count'),
    AvgFollowers=('Followers','mean'),
    AvgEngRate=('EngagementRate','mean'),
    TotalFees=('Fee','sum'),
    TotalRevenue=('Revenue','sum'),
    AvgROAS=('ROAS','mean'),
    AvgCostPerEng=('CostPerEng','mean')
).reindex(tier_order).round(2)
print(by_tier.to_string())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
tier_colors = [TEAL, PURPLE, AMBER, NAVY, RED]

axes[0].bar(tier_order, by_tier['AvgROAS'], color=tier_colors)
axes[0].axhline(y=1, color=RED, linestyle='--', linewidth=1.5, label='Break-even')
axes[0].set_title('Avg ROAS by Influencer Tier')
axes[0].legend()
for i, v in enumerate(by_tier['AvgROAS'].values):
    if not np.isnan(v):
        axes[0].text(i, v+0.05, f'{v:.2f}x', ha='center', fontsize=9)

axes[1].bar(tier_order, by_tier['AvgEngRate'], color=tier_colors)
axes[1].set_title('Avg Engagement Rate by Tier (%)')
for i, v in enumerate(by_tier['AvgEngRate'].values):
    if not np.isnan(v):
        axes[1].text(i, v+0.05, f'{v:.1f}%', ha='center', fontsize=9)

axes[2].scatter(influencers['Followers']/1000, influencers['EngagementRate'],
                c=influencers['Tier'].map(dict(zip(tier_order, tier_colors))),
                alpha=0.7, s=60)
axes[2].set_title('Followers vs Engagement Rate')
axes[2].set_xlabel('Followers (K)')
axes[2].set_ylabel('Engagement Rate (%)')
legend_patches = [mpatches.Patch(color=c, label=t) for t,c in zip(tier_order, tier_colors)]
axes[2].legend(handles=legend_patches, fontsize=8)

plt.tight_layout()
plt.savefig('chart_09_influencer_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Churn Prediction Model

**Definition:** Customer is churned if no order in last 90 days (as of 2024-12-31).  
**Model:** Random Forest Classifier  
**Output:** Predictions exported back to SQLite for Power BI consumption.


In [ ]:
REFERENCE_DATE = pd.Timestamp('2024-12-31')
CHURN_DAYS = 90

cust_agg = orders.groupby('CustomerID').agg(
    TotalOrders=('OrderID','count'),
    TotalRevenue=('Revenue','sum'),
    AvgOrderValue=('Revenue','mean'),
    LastOrderDate=('OrderDate','max'),
    FirstOrderDate=('OrderDate','min'),
    TotalReturns=('IsReturn','sum')
).reset_index()

cust_agg['DaysSinceLastOrder'] = (REFERENCE_DATE - cust_agg['LastOrderDate']).dt.days
cust_agg['CustomerAge']        = (REFERENCE_DATE - cust_agg['FirstOrderDate']).dt.days
cust_agg['IsChurned']          = (cust_agg['DaysSinceLastOrder'] > CHURN_DAYS).astype(int)

model_df = customers.merge(cust_agg, on='CustomerID', how='inner')

print(f"Total customers with orders: {len(model_df)}")
print(f"Churned:  {model_df['IsChurned'].sum()} ({model_df['IsChurned'].mean()*100:.1f}%)")
print(f"Active:   {(model_df['IsChurned']==0).sum()} ({(model_df['IsChurned']==0).mean()*100:.1f}%)")


In [ ]:
le_channel = LabelEncoder()
le_segment = LabelEncoder()
le_gender  = LabelEncoder()

model_df['AcqChannel_enc'] = le_channel.fit_transform(model_df['AcqChannel'])
model_df['Segment_enc']    = le_segment.fit_transform(model_df['Segment'])
model_df['Gender_enc']     = le_gender.fit_transform(model_df['Gender'])

FEATURES = ['DaysSinceLastOrder','TotalOrders','TotalRevenue','AvgOrderValue',
            'CustomerAge','Age','AcqChannel_enc','Segment_enc','TotalReturns']

X = model_df[FEATURES].fillna(0)
y = model_df['IsChurned']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

y_pred      = rf.predict(X_test)
y_pred_prob = rf.predict_proba(X_test)[:,1]

print('=== MODEL PERFORMANCE ===')
print(classification_report(y_test, y_pred, target_names=['Active','Churned']))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_pred_prob):.3f}')


In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].barh(importances.index, importances.values, color=PURPLE)
axes[0].set_title('Feature Importance — Churn Model')
axes[0].set_xlabel('Importance Score')
for i, v in enumerate(importances.values):
    axes[0].text(v+0.001, i, f'{v:.3f}', va='center', fontsize=9)

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='RdYlGn',
            xticklabels=['Active','Churned'],
            yticklabels=['Active','Churned'], ax=axes[1])
axes[1].set_title('Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('chart_10_churn_model.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Export predictions back to SQLite for Power BI
all_pred_prob = rf.predict_proba(X.fillna(0))[:,1]
all_pred      = rf.predict(X.fillna(0))

model_df['ChurnProbability'] = all_pred_prob.round(4)
model_df['ChurnPrediction']  = all_pred
model_df['ChurnRisk'] = pd.cut(
    model_df['ChurnProbability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

churn_export = model_df[[
    'CustomerID','ChurnProbability','ChurnPrediction','ChurnRisk',
    'DaysSinceLastOrder','TotalOrders','TotalRevenue','AvgOrderValue'
]].copy()
churn_export['ChurnRisk'] = churn_export['ChurnRisk'].astype(str)

conn = sqlite3.connect(DB_PATH)
churn_export.to_sql('Churn_Predictions', conn, if_exists='replace', index=False)
conn.close()

print(f'Predictions exported: {len(churn_export)} customers')
print()
print('Risk Distribution:')
print(model_df['ChurnRisk'].value_counts().to_string())
print()
print('Top 10 highest churn risk:')
top10 = model_df[['CustomerID','ChurnProbability','ChurnRisk','TotalOrders','TotalRevenue']].sort_values('ChurnProbability', ascending=False).head(10)
print(top10.to_string(index=False))


## 11. Key Findings Summary

In [ ]:
print('='*60)
print('SOCIAL MEDIA ANALYTICS — EDA KEY FINDINGS')
print('='*60)

total_rev    = orders['Revenue'].sum()
total_margin = orders['Margin'].sum()
top_platform = platform.sort_values('TotalReach', ascending=False).index[0]
top_eng_plat = platform.sort_values('AvgEngRate', ascending=False).index[0]
top_content  = content.sort_values('AvgEngRate', ascending=False).index[0]
top_channel  = by_channel.sort_values('CLV', ascending=False).index[0]
top_tier     = by_tier.sort_values('AvgROAS', ascending=False).index[0]
high_risk    = (model_df['ChurnRisk']=='High Risk').sum()
churn_pct    = model_df['IsChurned'].mean()*100
overall_roas = camp_agg['TotalRevenue'].sum()/camp_agg['TotalSpend'].sum()

print(f"PLATFORM & CONTENT:")
print(f"  Top Platform by Reach:     {top_platform}")
print(f"  Top Platform by Engagement:{top_eng_plat}")
print(f"  Best Content Type:         {top_content}")
print(f"  Total Posts Published:     {len(posts):,}")
print(f"  Total Reach Generated:     {posts['Reach'].sum()/1e6:.1f}M")
print()
print(f"CAMPAIGNS:")
print(f"  Total Campaigns:           {len(camps)}")
print(f"  Overall ROAS:              {overall_roas:.2f}x")
print(f"  Above Break-even:          {(camp_agg['ROAS']>=1).sum()}")
print(f"  Below Break-even:          {(camp_agg['ROAS']<1).sum()}")
print()
print(f"REVENUE & CUSTOMERS:")
print(f"  Total Revenue:             EUR {total_rev:,.0f}")
print(f"  Total Margin:              EUR {total_margin:,.0f}")
print(f"  Margin Rate:               {total_margin/total_rev*100:.1f}%")
print(f"  Best Channel by CLV:       {top_channel}")
print(f"  Total Customers:           {len(customers)}")
print()
print(f"INFLUENCERS:")
print(f"  Best Tier by ROAS:         {top_tier}")
print(f"  Total Partners:            {len(influencers)}")
print()
print(f"CHURN MODEL:")
print(f"  Current Churn Rate:        {churn_pct:.1f}%")
print(f"  High Risk Customers:       {high_risk}")
print(f"  ROC-AUC Score:             {roc_auc_score(y_test, y_pred_prob):.3f}")
print(f"  Predictions in SQLite:     Ready for Power BI")
print('='*60)
